In [1]:
from openai import OpenAI
import json
from datasets import load_dataset
import langid
from tqdm.auto import trange
import numpy as np

In [2]:
client = OpenAI(base_url="http://10.167.31.201:11434/v1", api_key="ollama")  # Dummy key

In [9]:
label2code = {
    "Spanish": "es",
    "English": "en",
    "French": "fr",
    "German": "de",
    "Chinese": "zh",
}


def lang_classification(text, label):
    lang, confidence = langid.classify(text)
    return int(lang == label2code[label])

In [60]:
def instruction_following_evaluation(
    dataset, thinking_responses, final_answer_responses
):
    list_results = []
    for idx in trange(len(dataset["test"])):
        # for idx in trange(3):
        thinking_response = thinking_responses[idx]["response"]
        final_answer_response = final_answer_responses[idx]["response"]
        intr_eval = dataset["test"][idx]["intr_eval"]
        instruction_type = dataset["test"][idx]["instruction_type"]
        if instruction_type == "bilingual_reasoning":
            thinking_lang = intr_eval.split("->")[0].strip()
            final_lang = intr_eval.split("->")[1].strip()
            thinking_eval = lang_classification(thinking_response, thinking_lang)
            final_eval = lang_classification(final_answer_response, final_lang)
            list_results.append((thinking_eval + final_eval) / 2)
        else:
            response = client.chat.completions.create(
                model="gpt-oss:120b",
                messages=[
                    {
                        "role": "system",
                        "content": "Reasoning: low",
                    },
                    {
                        "role": "user",
                        "content": f"{thinking_response} {intr_eval}. Say 'Yes' or 'No' only.",
                    },
                ],
            )
            if "yes" in response.choices[0].message.content.lower():
                list_results.append(1)
            elif "no" in response.choices[0].message.content.lower():
                list_results.append(0)
            else:
                print("Error in response:", response.choices[0].message.content)
    print("Accuracy:", np.mean(list_results))
    return np.mean(list_results)

In [53]:
def extract_answer(question, answer):
    system_prompt = "The user provides a question and an answer. The question is always a boolean question, i.e., the answers can be yes or no. The answer provided by the user may be long, non-english, and wrong, that's fine. Your task is to extract 'yes' or 'no' from the answer. If the answer is not clear, you can make your best guess. Respond with 'Yes' or 'No' only."
    response = client.chat.completions.create(
        model="gpt-oss:120b",
        messages=[
            {
                "role": "system",
                "content": f"{system_prompt}\nReasoning: low",
            },
            {
                "role": "user",
                "content": f"Question:{question}\nAnswer:{answer}.",
            },
        ],
    )
    return response.choices[0].message.content.lower()

In [ ]:
def evaluate_accuracy(dataset, final_answer_responses):
    list_results = []
    for i in trange(len(dataset["test"])):
        question = dataset["test"][i]["question"]
        answer = final_answer_responses[i]["response"]
        extracted_answer = extract_answer(question, answer)
        if "yes" in extracted_answer:
            list_results.append(1)
        elif "no" in extracted_answer:
            list_results.append(0)
        else:
            print("Error in response:", extracted_answer)
    print("Accuracy:", np.mean(list_results))
    return np.mean(list_results)

In [61]:
def strategyqa_evaluation(thinking_path, final_ans_path):
    dataset = load_dataset("haritzpuerto/strategyqa-icot-test-set")
    thinking_responses = []
    with open(thinking_path, "r") as f:
        for line in f:
            thinking_responses.append(json.loads(line))
    final_answer_responses = []
    with open(final_ans_path, "r") as f:
        for line in f:
            final_answer_responses.append(json.loads(line))

    # instruction following evaluation
    if_acc = instruction_following_evaluation(
        dataset, thinking_responses, final_answer_responses
    )
    # reasoning performance evaluation
    reasoning_acc = evaluate_accuracy(dataset, final_answer_responses)
    return {"IF_acc": if_acc, "reasoning_acc": reasoning_acc}

In [ ]:
thinking_path = "/Users/haritz/Projects/ifr/models/DeepSeek-R1-Distill-Llama-8B/unsloth/gsm8k-icot--gpstoss120-2k/sft/5e-5/20251022-154541/haritzpuerto/strategyqa-icot-test-set/10000_tokens/responses_thinking.jsonl"
final_ans_path = "/Users/haritz/Projects/ifr/models/DeepSeek-R1-Distill-Llama-8B/unsloth/gsm8k-icot--gpstoss120-2k/sft/5e-5/20251022-154541/haritzpuerto/strategyqa-icot-test-set/10000_tokens/responses_final_ans.jsonl"
evaluation_results = strategyqa_evaluation(thinking_path, final_ans_path)
# print the dictionary as a table
for key, value in evaluation_results.items():
    print(f"{key}: {value}")